# Экзаменационная задача №04
## Выбор ценовой стратегии для AI-сервиса по созданию контента

**Компания:** «AI-Копирайтер»  
**Дисциплина:** Экономика информатизации  
**Студент:** Нахныков А.А.  
**Группа:** ПИ 4-3

---

### Цель работы

Сравнить четыре ценовые стратегии AI-сервиса, рассчитать ARPU, LTV, LTV/CAC, спрогнозировать финансовые результаты на три года, определить NPV, провести анализ чувствительности и выбрать стратегию выхода на рынок.

Расчёты выполняются на Python в Jupyter Notebook.

## 1. Постановка задачи

Стартап «AI-Копирайтер» разработал сервис, который создаёт маркетинговые, SEO-тексты и публикации для социальных сетей с помощью искусственного интеллекта.

Необходимо:

1. Рассчитать ARPU и LTV для каждой стратегии.
2. Сравнить LTV/CAC по трём каналам привлечения.
3. Построить прогноз пользовательской базы и финансовых результатов на три года.
4. Рассчитать NPV каждой стратегии.
5. Провести анализ чувствительности к конверсии, Churn и CAC.
6. Выбрать наиболее подходящую стратегию.
7. Сформулировать план внедрения, метрики и меры снижения рисков.

## 2. Исходные данные

### 2.1. Рынок и затраты

| Параметр | Значение |
|---|---:|
| Потенциальный рынок | 500 000 пользователей |
| Рост рынка | 25% в год |
| Затраты на разработку | 15 000 000 руб. |
| Поддержание и облачные услуги | 2 000 000 руб./мес. |
| CAC: реклама | 1 500 руб. |
| CAC: контент-маркетинг и SEO | 800 руб. |
| CAC: виральный рост | 100 руб. |
| Ставка дисконтирования | 12% годовых |
| Горизонт планирования | 3 года |

### 2.2. Ценовые стратегии

| Стратегия | Цена / средний чек | Конверсия | Churn в месяц |
|---|---:|---:|---:|
| Freemium | 2 000 руб. | 2% | 8% |
| Подписка | 1 500 руб. | 20% | 5% |
| Динамическая | 1 200 руб. | 15% | 6% |
| Персонализированная | 1 500 руб. | 12% | 4% |

**Примечание:** для динамической и персонализированной стратегий в расчётах используется указанный средний чек. Churn трактуется как ежемесячный отток, поскольку ARPU рассчитывается за месяц, а LTV = ARPU / Churn.

### 2.3. Допущения финансовой модели

- Начальная база в год 0 — 1 000 пользователей.
- Год 1: база увеличивается на 100% — до 2 000 пользователей.
- Год 2: рост на 50% — до 3 000 пользователей.
- Год 3: рост на 20% — до 3 600 пользователей.
- Базовый канал привлечения для прогноза прибыли — контент-маркетинг и SEO (CAC = 800 руб.).
- CAC оплачивается за каждого нового пользователя: 1 000 в год 0, 1 000 в год 1, 1 000 в год 2 и 600 в год 3.
- Затраты на разработку учитываются в год 0.
- Поддержание сервиса составляет 24 млн руб. за каждый год 1–3.
- Налог на прибыль не рассчитывается, поскольку налоговая база и режим налогообложения в условии не заданы.

**Методическое замечание:** прогнозная база пользователей используется как плановая база привлечённых пользователей. Для оценки выручки она корректируется на среднюю долю пользователей, сохранившихся в течение года с учётом ежемесячного Churn. Это упрощённое допущение модели.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Общие параметры проекта
market_size = 500_000
market_growth = 0.25
development_cost = 15_000_000
monthly_maintenance = 2_000_000
annual_maintenance = monthly_maintenance * 12

discount_rate = 0.12
planning_years = 3

# Стоимость привлечения пользователя по каналам
cac_channels = {
    "Реклама": 1_500,
    "Контент-маркетинг и SEO": 800,
    "Виральный рост": 100
}

# Данные ценовых стратегий
strategies = {
    "Freemium": {"price": 2_000, "conversion": 0.02, "churn": 0.08},
    "Подписка": {"price": 1_500, "conversion": 0.20, "churn": 0.05},
    "Динамическая": {"price": 1_200, "conversion": 0.15, "churn": 0.06},
    "Персонализированная": {"price": 1_500, "conversion": 0.12, "churn": 0.04}
}

# Плановая пользовательская база по годам
users_by_year = {0: 1_000, 1: 2_000, 2: 3_000, 3: 3_600}
new_users_by_year = {0: 1_000, 1: 1_000, 2: 1_000, 3: 600}

print(f"Потенциальный рынок: {market_size:,} пользователей")
print(f"Разработка: {development_cost:,} руб.")
print(f"Поддержание в год: {annual_maintenance:,} руб.")
print(f"Ставка дисконтирования: {discount_rate:.0%}")

## 3. Часть 1. Расчёт ключевых метрик

### 3.1. ARPU — средний доход на пользователя

ARPU показывает, сколько в среднем сервис получает за месяц от одного пользователя с учётом конверсии в платную версию.

Формула:

\[
ARPU = Цена \times Конверсия
\]

Например, для Freemium: 2 000 × 2% = 40 руб. в месяц. Это средний доход в расчёте на одного пользователя всей базы, а не цена платной подписки.

In [ ]:
arpu_values = {}

for name, data in strategies.items():
    arpu = data["price"] * data["conversion"]
    arpu_values[name] = arpu

arpu_df = pd.DataFrame([
    {
        "Стратегия": name,
        "Цена / средний чек, руб.": data["price"],
        "Конверсия": f'{data["conversion"]:.0%}',
        "ARPU, руб./мес.": arpu_values[name]
    }
    for name, data in strategies.items()
])

display(arpu_df)

### 3.2. LTV — пожизненная ценность клиента

LTV показывает, какой доход в среднем приносит пользователь за всё время взаимодействия с сервисом.

По условию используется формула:

\[
LTV = \frac{ARPU}{Churn}
\]

Срок жизни клиента рассчитывается как 1 / Churn. Например, при Churn 5% срок жизни равен 20 месяцам.

Важно: в этой учебной модели LTV рассчитывается по выручке, без вычета расходов на поддержку и обслуживание.

In [ ]:
ltv_values = {}
lifetime_values = {}

for name, data in strategies.items():
    ltv_values[name] = arpu_values[name] / data["churn"]
    lifetime_values[name] = 1 / data["churn"]

ltv_df = pd.DataFrame([
    {
        "Стратегия": name,
        "ARPU, руб./мес.": arpu_values[name],
        "Churn": f'{strategies[name]["churn"]:.0%}',
        "Срок жизни, мес.": lifetime_values[name],
        "LTV, руб.": ltv_values[name]
    }
    for name in strategies
])

display(ltv_df)

### 3.3. Соотношение LTV/CAC

CAC — стоимость привлечения одного пользователя. LTV/CAC показывает, сколько рублей дохода за жизненный цикл приходится на каждый рубль затрат на привлечение.

\[
LTV/CAC = \frac{LTV}{CAC}
\]

Ориентир для оценки — значение выше 3x. Чем выше показатель, тем эффективнее привлечение. Однако он не учитывает все постоянные расходы бизнеса.

In [ ]:
ltv_cac_rows = []

for strategy, ltv in ltv_values.items():
    for channel, cac in cac_channels.items():
        ltv_cac_rows.append({
            "Стратегия": strategy,
            "Канал": channel,
            "CAC, руб.": cac,
            "LTV, руб.": ltv,
            "LTV/CAC": ltv / cac
        })

ltv_cac_df = pd.DataFrame(ltv_cac_rows)
display(ltv_cac_df.pivot(index="Стратегия", columns="Канал", values="LTV/CAC").round(2))

**Вывод по метрикам**

- Подписка имеет наибольший ARPU — 300 руб./мес. и LTV — 6 000 руб.
- Персонализированная стратегия имеет самый низкий Churn — 4%, поэтому срок жизни клиента самый длинный.
- Виральный канал даёт лучшие значения LTV/CAC, так как привлечение стоит всего 100 руб.
- При выборе канала важно учитывать не только LTV/CAC, но и реальную возможность обеспечить нужный объём привлечения.

## 4. Часть 2. Прогноз пользовательской базы и финансовых результатов

### 4.1. Прогноз пользовательской базы

По условию база составляет 1 000 пользователей в год 0. Затем применяется заданный прирост: 100%, 50% и 20%.

Количество новых пользователей за год — это разница между базой текущего и предыдущего года.

In [ ]:
users_df = pd.DataFrame({
    "Год": list(users_by_year.keys()),
    "Прирост": ["—", "100%", "50%", "20%"],
    "Всего пользователей": list(users_by_year.values()),
    "Новые пользователи": list(new_users_by_year.values())
})

display(users_df)

### 4.2. Учёт Churn при оценке выручки

Поскольку Churn задан в месяц, часть пользователей уходит в течение года. Для упрощённой оценки используется средний коэффициент удержания за 12 месяцев:

\[
K_{удержания} = \frac{1 + (1-Churn) + ... + (1-Churn)^{11}}{12}
\]

Средняя активная база равна плановой базе пользователей, умноженной на этот коэффициент.

Годовая выручка:

\[
Выручка = Средняя\ активная\ база \times ARPU \times 12
\]

In [ ]:
def average_retention_factor(monthly_churn):
    monthly_retention = 1 - monthly_churn
    return sum(monthly_retention ** month for month in range(12)) / 12

retention_df = pd.DataFrame([
    {
        "Стратегия": name,
        "Churn в месяц": data["churn"],
        "Средний коэффициент удержания за год": average_retention_factor(data["churn"])
    }
    for name, data in strategies.items()
])

display(retention_df.style.format({
    "Churn в месяц": "{:.0%}",
    "Средний коэффициент удержания за год": "{:.3f}"
}))

### 4.3. Расчёт прибыли по стратегиям

Для каждой стратегии рассчитывается годовая выручка. Затем вычитаются:

- расходы на поддержание сервиса — 24 млн руб. в год;
- расходы на привлечение новых пользователей — число новых пользователей × CAC;
- первоначальные затраты на разработку — 15 млн руб. в год 0.

Для базового финансового сценария используется контент-маркетинг с CAC = 800 руб.

Прибыль здесь означает упрощённый финансовый результат до налогообложения. Налог не рассчитывается, поскольку его параметры в условии не заданы.

In [ ]:
base_cac = cac_channels["Контент-маркетинг и SEO"]

def build_financial_model(strategy_name, conversion_factor=1.0, churn_factor=1.0, cac_factor=1.0, cac_value=base_cac):
    data = strategies[strategy_name]
    adjusted_conversion = min(data["conversion"] * conversion_factor, 1.0)
    adjusted_churn = min(max(data["churn"] * churn_factor, 0.0), 0.99)
    adjusted_arpu = data["price"] * adjusted_conversion

    cash_flows = []
    rows = []

    for year in range(4):
        users = users_by_year[year]
        new_users = new_users_by_year[year]
        acquisition_cost = new_users * cac_value * cac_factor

        if year == 0:
            revenue = 0
            maintenance = 0
            development = development_cost
        else:
            retention = average_retention_factor(adjusted_churn)
            average_active_users = users * retention
            revenue = average_active_users * adjusted_arpu * 12
            maintenance = annual_maintenance
            development = 0

        profit = revenue - acquisition_cost - maintenance - development
        cash_flows.append(profit)

        rows.append({
            "Год": year,
            "Плановая база": users,
            "Новые пользователи": new_users,
            "Выручка, руб.": revenue,
            "Привлечение, руб.": acquisition_cost,
            "Поддержание, руб.": maintenance,
            "Разработка, руб.": development,
            "Финансовый результат, руб.": profit
        })

    return pd.DataFrame(rows), cash_flows

financial_models = {}
for name in strategies:
    model_df, cash_flows = build_financial_model(name)
    financial_models[name] = {"table": model_df, "cash_flows": cash_flows}
    print(f"Стратегия: {name}")
    display(model_df.style.format({
        col: "{:,.0f}" for col in model_df.columns if col != "Год"
    }))

## 5. NPV — чистая приведённая стоимость

NPV показывает, сколько проект создаёт или теряет в сегодняшней стоимости с учётом ставки дисконтирования.

\[
NPV = CF_0 + \sum_{t=1}^{T}\frac{CF_t}{(1+r)^t}
\]

Если NPV больше нуля, проект финансово привлекателен в рамках заданных допущений. Если меньше нуля — прогнозируемых поступлений недостаточно для покрытия затрат с учётом ставки дисконтирования.

In [ ]:
def calculate_npv(cash_flows, rate=discount_rate):
    return sum(flow / (1 + rate) ** year for year, flow in enumerate(cash_flows))

npv_results = []

for name, model in financial_models.items():
    npv = calculate_npv(model["cash_flows"])
    npv_results.append({"Стратегия": name, "NPV, руб.": npv})

npv_df = pd.DataFrame(npv_results).sort_values("NPV, руб.", ascending=False)
display(npv_df.style.format({"NPV, руб.": "{:,.0f}"}))

### Визуализация NPV

График позволяет быстро сравнить стратегии. Отрицательное значение означает, что в рамках полной модели проект не окупает все учтённые затраты за три года.

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(npv_df["Стратегия"], npv_df["NPV, руб."] / 1_000_000)
plt.axhline(0, linewidth=1)
plt.title("NPV ценовых стратегий за 3 года")
plt.ylabel("NPV, млн руб.")
plt.xlabel("Ценовая стратегия")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.show()

### Важное сопоставление с кратким решением для преподавателя

В кратком решении указаны NPV: Freemium −18,2 млн руб., подписка +4,8 млн руб., динамическая −5,4 млн руб., персонализированная +2,8 млн руб.

Эти значения не получается однозначно воспроизвести из приведённых исходных данных: в расчёте не показано, как учтены 15 млн руб. на разработку, затраты на привлечение пользователей в каждом году и отток пользователей. В кратком примере годовой прогноз также использует всю базу как платящую без корректировки на Churn.

В текущей модели все указанные расходы отражены явно. Поэтому её NPV может отличаться от результата в кратком решении. Это не ошибка Python, а следствие разных допущений. Для честной оценки стартапа затраты на разработку и поддержку нельзя игнорировать.

## 6. Часть 3. Анализ чувствительности

Проверим, как изменится NPV при изменении трёх ключевых параметров на ±30%:

- конверсия: влияет на ARPU и выручку;
- Churn: влияет на среднюю активную базу и удержание;
- CAC: влияет на расходы на привлечение.

Для сопоставимости базовым каналом остаётся контент-маркетинг (CAC = 800 руб.).

In [ ]:
sensitivity_rows = []

for strategy_name in strategies:
    base_npv = calculate_npv(build_financial_model(strategy_name)[1])

    scenarios = [
        ("Конверсия", "+30%", {"conversion_factor": 1.3}),
        ("Конверсия", "-30%", {"conversion_factor": 0.7}),
        ("Churn", "+30%", {"churn_factor": 1.3}),
        ("Churn", "-30%", {"churn_factor": 0.7}),
        ("CAC", "+30%", {"cac_factor": 1.3}),
        ("CAC", "-30%", {"cac_factor": 0.7})
    ]

    for parameter, change, kwargs in scenarios:
        _, scenario_flows = build_financial_model(strategy_name, **kwargs)
        scenario_npv = calculate_npv(scenario_flows)
        sensitivity_rows.append({
            "Стратегия": strategy_name,
            "Параметр": parameter,
            "Изменение": change,
            "NPV, руб.": scenario_npv,
            "Изменение NPV, руб.": scenario_npv - base_npv
        })

sensitivity_df = pd.DataFrame(sensitivity_rows)
display(sensitivity_df.style.format({
    "NPV, руб.": "{:,.0f}",
    "Изменение NPV, руб.": "{:+,.0f}"
}))

### 6.1. Визуализация чувствительности

Для каждой стратегии сравним, насколько меняется NPV при изменении каждого параметра. Чем больше абсолютное изменение NPV, тем чувствительнее стратегия к этому параметру.

In [ ]:
for strategy_name in strategies:
    subset = sensitivity_df[sensitivity_df["Стратегия"] == strategy_name].copy()
    subset["Сценарий"] = subset["Параметр"] + " " + subset["Изменение"]

    plt.figure(figsize=(9, 4))
    plt.bar(subset["Сценарий"], subset["Изменение NPV, руб."] / 1_000_000)
    plt.axhline(0, linewidth=1)
    plt.title(f"Чувствительность NPV: {strategy_name}")
    plt.ylabel("Изменение NPV, млн руб.")
    plt.xticks(rotation=35, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

### 6.2. Определение устойчивости стратегий

Устойчивость оценивается по максимальному абсолютному изменению NPV при заданных колебаниях конверсии, Churn и CAC. Чем меньше такое изменение, тем менее чувствительна стратегия к изменению параметров.

Дополнительно учитывается сам базовый NPV: стратегия с небольшим изменением, но сильно отрицательным NPV не становится автоматически привлекательной.

In [ ]:
sensitivity_summary = (
    sensitivity_df
    .assign(Абсолютное_изменение=lambda x: x["Изменение NPV, руб."].abs())
    .groupby("Стратегия")["Абсолютное_изменение"]
    .max()
    .reset_index(name="Максимальное изменение NPV, руб.")
    .merge(npv_df, on="Стратегия")
    .sort_values("Максимальное изменение NPV, руб.")
)

display(sensitivity_summary.style.format({
    "Максимальное изменение NPV, руб.": "{:,.0f}",
    "NPV, руб.": "{:,.0f}"
}))

## 7. Часть 4. Стратегические рекомендации

### 7.1. Выбор ценовой стратегии

По расчётным метрикам наиболее сильной выглядит **подписка**:

- самый высокий ARPU — 300 руб. в месяц;
- самый высокий LTV — 6 000 руб.;
- низкий Churn — 5% в месяц;
- высокие значения LTV/CAC, особенно при контент-маркетинге и виральном росте.

Но окончательный выбор должен учитывать NPV. Если NPV отрицательный, это означает, что при заданных расходах и размере пользовательской базы бизнес-модель требует доработки. Подписка остаётся наиболее перспективной для тестирования, но запускать её в полном масштабе без проверки экономики нельзя.

Рекомендация: выбрать подписку как основную модель монетизации, запустить ограниченный пилот и одновременно снижать постоянные расходы и CAC. Дополнительно можно использовать бесплатный ограниченный тариф как канал привлечения, но не как основную модель дохода.

### 7.2. План внедрения

**Этап 1. Подготовка и запуск (первые 3 месяца)**

- Запустить подписку с бесплатным пробным периодом.
- Проверить готовность пользователей платить 1 500 руб. в месяц.
- Подготовить примеры текстов, демонстрации и обучающие материалы.
- Начать продвижение через SEO, блог и профессиональные сообщества.

**Этап 2. Рост (4–12 месяцев)**

- Развивать реферальную программу.
- Добавить шаблоны под разные отрасли и задачи.
- Проводить A/B-тестирование цены и пробного периода.
- Анализировать причины отказа от подписки.

**Этап 3. Масштабирование (2–3 год)**

- Улучшать качество генерации и скорость сервиса.
- Добавлять интеграции с CMS и маркетинговыми платформами.
- Тестировать тарифы для команд и компаний.
- Оптимизировать расходы на облачную инфраструктуру.

### 7.3. Коммуникационная стратегия

Основное сообщение для рынка: сервис помогает быстрее создавать маркетинговые материалы и экономить время специалистов.

Ключевые каналы:

- SEO-статьи и полезные материалы о создании контента;
- видео-демонстрации сервиса и разборы примеров;
- публикации в сообществах маркетологов и SMM-специалистов;
- бесплатный пробный период;
- реферальная программа для действующих пользователей.

Коммуникация должна показывать конкретную пользу: скорость подготовки текста, удобство редактирования и снижение трудозатрат.

### 7.4. Ключевые метрики мониторинга

| Метрика | Целевой ориентир | Частота |
|---|---:|---|
| Конверсия в оплату | не ниже 20% | ежемесячно |
| Churn | не выше 5% в месяц | ежемесячно |
| LTV/CAC | выше 3x | ежеквартально |
| MRR — регулярная месячная выручка | устойчивый рост | ежемесячно |
| CAC | снижение по каналам | ежемесячно |
| Доля пользователей, завершивших пробный период с оплатой | рост | ежемесячно |
| Доступность сервиса | не ниже 99,5% | ежемесячно |

Целевые значения являются ориентирами для пилота, а не гарантированными результатами.

### 7.5. Риски и меры снижения

| Риск | Возможное последствие | Меры снижения |
|---|---|---|
| Высокий Churn | Пользователи быстро уходят | Улучшать качество текстов, собирать обратную связь |
| Низкая конверсия | Недостаточная выручка | Тестировать тарифы и пробный период |
| Рост CAC | Дорогое привлечение | Развивать SEO, рекомендации и партнёрства |
| Сильная конкуренция | Давление на цену | Развивать специализацию и уникальные функции |
| Рост облачных затрат | Снижение маржи | Оптимизировать модели и лимиты генерации |
| Ошибки AI-контента | Репутационные и юридические риски | Проверка фактов, предупреждения и модерация |
| Изменение законодательства | Дополнительные требования | Юридический мониторинг и защита данных |

## 8. Итоговые выводы

1. Подписка показывает лучшие значения ARPU и LTV среди четырёх рассмотренных стратегий.
2. Виральный рост имеет минимальный CAC и поэтому обеспечивает наиболее высокое LTV/CAC.
3. Персонализированная модель выигрывает по удержанию пользователей благодаря самому низкому Churn.
4. Полная финансовая модель должна учитывать 15 млн руб. на разработку, ежегодные расходы на поддержку и затраты на привлечение новых пользователей.
5. При заданных постоянных расходах и небольшой начальной базе NPV может оставаться отрицательным даже при хороших продуктовых метриках. Это сигнал о необходимости масштабирования, снижения затрат или изменения цены.
6. Рекомендуется тестировать подписку как основную стратегию, привлекать пользователей через контент-маркетинг и постепенно развивать виральный канал.

**Общий вывод:** лучшая ценовая стратегия определяется не только ценой и конверсией. Необходимо одновременно учитывать удержание пользователей, стоимость привлечения и постоянные расходы на работу AI-сервиса.